# Caso de uso: sistema de recomendación simple en e-commerce

## Objetivo
Construir un sistema de recomendación sencillo y explicable para una tienda en línea.  
El sistema usará dos enfoques:

1. **Popularidad**: recomendar los productos más consumidos.
2. **Filtrado colaborativo item-item**: recomendar productos similares a los ya consumidos por un usuario.

Este cuaderno está diseñado para ejecutarse directamente en **Google Colab**.


## Contexto del caso
Supongamos una tienda digital que vende productos de tecnología, educación, oficina y accesorios.  
Queremos responder una pregunta de negocio simple:

> **¿Qué productos deberíamos recomendar a cada usuario a partir de su historial de interacción?**

Para ello usaremos un dataset pequeño y comprensible, ideal para fines académicos y demostrativos.


## Bloque 1. Librerías
En este bloque importamos las librerías necesarias:

- **pandas**: manipulación de datos.
- **numpy**: operaciones numéricas.
- **matplotlib**: visualización.
- **sklearn**: cálculo de similitud por coseno.

Estas librerías suelen funcionar sin problemas en Colab.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_columns", None)


## Bloque 2. Cargar el dataset
Aquí cargamos el archivo CSV.  
Si abres este notebook en Colab, puedes subir el archivo manualmente o leerlo desde Google Drive.

El dataset contiene:

- **user_id**: identificador del usuario.
- **item_id**: identificador del producto.
- **rating**: valoración o intensidad de preferencia.
- **timestamp**: momento de la interacción.
- **item_name**: nombre del producto.
- **category**: categoría del producto.


In [ ]:
# Si trabajas en Colab, sube primero el archivo dataset_recomendacion_ecommerce.csv
df = pd.read_csv("dataset_recomendacion_ecommerce.csv", parse_dates=["timestamp"])
df.head()


## Bloque 3. Exploración inicial
En este paso revisamos:

- tamaño del dataset,
- tipos de datos,
- número de usuarios,
- número de productos.

Esto sirve para entender el problema antes de modelar.


In [ ]:
print("Dimensión del dataset:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nUsuarios únicos:", df["user_id"].nunique())
print("Productos únicos:", df["item_id"].nunique())

df.describe(include="all")


## Bloque 4. Análisis exploratorio
Ahora observamos qué categorías aparecen con mayor frecuencia y cuáles son los productos más consumidos.

Esto ayuda a identificar sesgos de popularidad y posibles patrones de consumo.


In [ ]:
print("Interacciones por categoría:")
print(df["category"].value_counts())

print("\nTop 10 productos por número de interacciones:")
print(df["item_name"].value_counts().head(10))


In [ ]:
df["category"].value_counts().plot(kind="bar", title="Interacciones por categoría")
plt.xlabel("Categoría")
plt.ylabel("Frecuencia")
plt.show()


## Bloque 5. Preparación de datos
Para construir un recomendador de tipo colaborativo necesitamos una **matriz usuario-item**.

- Filas: usuarios
- Columnas: productos
- Valores: rating promedio o intensidad de interacción

Si un usuario interactuó varias veces con un producto, usamos el promedio.


In [ ]:
user_item = df.pivot_table(
    index="user_id",
    columns="item_id",
    values="rating",
    aggfunc="mean",
    fill_value=0
)

user_item.head()


## Bloque 6. Recomendador por popularidad
Este es el enfoque más simple.

### Idea
1. Contar qué productos reciben más interacciones.
2. Excluir los productos ya vistos por el usuario.
3. Recomendar los más populares restantes.

### Ventaja
Es un modelo robusto y útil para usuarios nuevos.

### Limitación
No personaliza tanto como otros enfoques.


In [ ]:
popularidad = (
    df.groupby(["item_id", "item_name"], as_index=False)
      .size()
      .rename(columns={"size": "num_interacciones"})
      .sort_values("num_interacciones", ascending=False)
)

popularidad.head(10)


In [ ]:
def recomendar_populares(user_id, df, top_n=5):
    vistos = set(df[df["user_id"] == user_id]["item_id"].unique())

    ranking = (
        df.groupby(["item_id", "item_name"], as_index=False)
          .size()
          .rename(columns={"size": "num_interacciones"})
          .sort_values("num_interacciones", ascending=False)
    )

    recomendaciones = ranking[~ranking["item_id"].isin(vistos)].head(top_n)
    return recomendaciones

recomendar_populares(user_id=3, df=df, top_n=5)


## Explicación de la función anterior
La función `recomendar_populares` hace lo siguiente:

- identifica qué productos ya vio el usuario,
- calcula un ranking global de popularidad,
- elimina del ranking los productos ya consumidos,
- devuelve los primeros `top_n`.

Es una línea base muy importante, porque cualquier modelo más avanzado debería superar este resultado.


## Bloque 7. Filtrado colaborativo item-item
Ahora construiremos un sistema un poco más inteligente.

### Idea
Si dos productos suelen ser consumidos por usuarios similares, entonces son parecidos.  
Por ejemplo:

- si muchos usuarios que compran un **Libro de Python** también consumen un **Curso de Machine Learning**,
- entonces ambos productos pueden recomendarse entre sí.

### Técnica
Usaremos **similitud del coseno** entre columnas de la matriz usuario-item.


In [ ]:
item_user_matrix = user_item.T

sim_matrix = cosine_similarity(item_user_matrix)
item_similarity = pd.DataFrame(
    sim_matrix,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

item_similarity.iloc[:5, :5]


## Bloque 8. Función de recomendación item-item
La lógica será:

1. tomar los productos con los que el usuario ya interactuó,
2. buscar productos similares a esos ítems,
3. sumar las similitudes,
4. excluir productos ya vistos,
5. ordenar de mayor a menor.

Así obtenemos recomendaciones personalizadas.


In [ ]:
item_id_to_name = df.drop_duplicates("item_id").set_index("item_id")["item_name"].to_dict()

def recomendar_item_item(user_id, user_item_matrix, item_similarity, top_n=5):
    # Productos ya consumidos por el usuario
    user_vector = user_item_matrix.loc[user_id]
    items_vistos = user_vector[user_vector > 0].index.tolist()

    scores = pd.Series(0, index=user_item_matrix.columns, dtype=float)

    for item in items_vistos:
        similitudes = item_similarity[item]
        scores = scores.add(similitudes, fill_value=0)

    # Eliminar ítems ya vistos
    scores = scores.drop(items_vistos, errors="ignore")

    # Ordenar de mayor a menor
    top_items = scores.sort_values(ascending=False).head(top_n)

    resultado = pd.DataFrame({
        "item_id": top_items.index,
        "score_similitud": top_items.values,
        "item_name": [item_id_to_name[i] for i in top_items.index]
    })

    return resultado[["item_id", "item_name", "score_similitud"]]

recomendar_item_item(user_id=3, user_item_matrix=user_item, item_similarity=item_similarity, top_n=5)


## Explicación del bloque anterior
Esta función produce recomendaciones personalizadas.

### ¿Qué representa el score?
El `score_similitud` no es una probabilidad.  
Es una medida acumulada de cercanía entre los productos que el usuario ya consumió y los productos candidatos.

### Interpretación
- score alto: el ítem candidato se parece mucho al historial del usuario.
- score bajo: el ítem tiene menor afinidad con sus preferencias.


## Bloque 9. Comparación entre ambos enfoques
Ahora comparamos para un mismo usuario:

- recomendación por popularidad,
- recomendación por similitud item-item.

Esto permite explicar la diferencia entre un sistema genérico y uno personalizado.


In [ ]:
usuario_ejemplo = 3

print("Historial del usuario:")
display(df[df["user_id"] == usuario_ejemplo][["item_name", "category", "rating"]].drop_duplicates())

print("\nRecomendación por popularidad:")
display(recomendar_populares(usuario_ejemplo, df, top_n=5))

print("\nRecomendación por item-item:")
display(recomendar_item_item(usuario_ejemplo, user_item, item_similarity, top_n=5))


## Bloque 10. Evaluación simple
Para evaluar de forma sencilla, usaremos una estrategia básica:

- para cada usuario, reservamos su **última interacción** como prueba,
- entrenamos con el resto,
- verificamos si el producto real aparece en el Top-K recomendado.

Mediremos **Hit Rate@K**, una métrica simple y muy intuitiva.


In [ ]:
df_sorted = df.sort_values(["user_id", "timestamp"]).copy()

test_idx = df_sorted.groupby("user_id").tail(1).index
test_df = df_sorted.loc[test_idx].copy()
train_df = df_sorted.drop(test_idx).copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)
test_df.head()


In [ ]:
train_user_item = train_df.pivot_table(
    index="user_id",
    columns="item_id",
    values="rating",
    aggfunc="mean",
    fill_value=0
)

train_item_user = train_user_item.T
train_sim = cosine_similarity(train_item_user)
train_item_similarity = pd.DataFrame(
    train_sim,
    index=train_item_user.index,
    columns=train_item_user.index
)

train_item_id_to_name = train_df.drop_duplicates("item_id").set_index("item_id")["item_name"].to_dict()


In [ ]:
def recomendar_item_item_train(user_id, train_user_item, train_item_similarity, top_n=5):
    if user_id not in train_user_item.index:
        return []

    user_vector = train_user_item.loc[user_id]
    items_vistos = user_vector[user_vector > 0].index.tolist()

    if len(items_vistos) == 0:
        return []

    scores = pd.Series(0, index=train_user_item.columns, dtype=float)

    for item in items_vistos:
        scores = scores.add(train_item_similarity[item], fill_value=0)

    scores = scores.drop(items_vistos, errors="ignore")
    return scores.sort_values(ascending=False).head(top_n).index.tolist()

def hit_rate_at_k(test_df, train_user_item, train_item_similarity, k=5):
    hits = 0
    total = 0

    for _, row in test_df.iterrows():
        user_id = row["user_id"]
        real_item = row["item_id"]

        recs = recomendar_item_item_train(user_id, train_user_item, train_item_similarity, top_n=k)
        if real_item in recs:
            hits += 1
        total += 1

    return hits / total if total > 0 else 0

for k in [3, 5]:
    print(f"Hit Rate@{k}: {hit_rate_at_k(test_df, train_user_item, train_item_similarity, k=k):.2f}")


## Interpretación de la evaluación
Si el **Hit Rate@5** es, por ejemplo, `0.58`, significa que en el 58% de los usuarios el producto real de prueba apareció dentro de las 5 recomendaciones.

En un caso académico, esto es suficiente para mostrar:

- cómo separar entrenamiento y prueba,
- cómo medir desempeño,
- cómo comparar modelos.

En entornos reales se suelen usar métricas adicionales como:

- Precision@K
- Recall@K
- MAP
- NDCG


## Bloque 11. Conclusiones del caso de uso

### Conclusiones técnicas
1. El recomendador por **popularidad** es la base más simple y útil.
2. El modelo **item-item** añade personalización sin requerir redes neuronales ni grandes recursos.
3. La construcción de la matriz usuario-item es el paso central del sistema.
4. La similitud del coseno permite encontrar relaciones entre productos de manera interpretable.

### Conclusiones de negocio
1. Un sistema simple puede aumentar exposición de productos relevantes.
2. Las recomendaciones personalizadas mejoran la experiencia del usuario.
3. Incluso con un dataset pequeño es posible demostrar el valor del enfoque.
4. Este tipo de prototipo puede evolucionar hacia un sistema híbrido o productivo.

### Posibles mejoras
- incorporar más usuarios y más productos,
- usar eventos implícitos como clics o compras,
- incluir variables de contenido del producto,
- agregar evaluación más robusta,
- desplegar el modelo en una API o dashboard.


## Cierre
Este cuaderno ofrece un caso de uso completo, ejecutable y explicable para clase, demostración o portafolio.

El flujo general fue:

1. cargar datos,  
2. explorar el dataset,  
3. construir una línea base por popularidad,  
4. construir un recomendador item-item,  
5. evaluar con una métrica simple,  
6. extraer conclusiones de negocio y técnicas.

Puedes extender este ejemplo con otros dominios como:
- películas,
- cursos,
- libros,
- productos de retail,
- plataformas educativas.
